# Customer Churn Analysis

Reproducible workflow: data quality → segmentation → predictive modelling → business interpretation.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score

df = pd.read_csv('../data/customer_churn_20000.csv')
display(df.head())
print('Shape:', df.shape)

In [ ]:
print('Missing values:', df.isna().sum().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Churn rate:', round(df['Churn'].mean()*100, 2), '%')

In [ ]:
contract = df.groupby('Contract').agg(Customers=('Customer_ID','count'), Churn_Rate=('Churn','mean'))
contract['Churn_Rate'] *= 100
internet = df.groupby('Internet_Service')['Churn'].mean().mul(100).sort_values(ascending=False)
satisfaction = df.groupby('Satisfaction_Score')['Churn'].mean().mul(100)
display(contract.sort_values('Churn_Rate', ascending=False)); display(internet); display(satisfaction)

In [ ]:
X = df.drop(columns=['Customer_ID','Churn']); y = df['Churn']
cat = ['Contract','Internet_Service']; num = [c for c in X.columns if c not in cat]
pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), cat), ('num', StandardScaler(), num)])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)
models={'Logistic Regression':LogisticRegression(max_iter=1000),'Random Forest':RandomForestClassifier(n_estimators=200,max_depth=10,random_state=42,n_jobs=-1)}
rows=[]
for name,model in models.items():
    pipe=Pipeline([('pre',pre),('model',model)]); pipe.fit(X_train,y_train)
    pred=pipe.predict(X_test); prob=pipe.predict_proba(X_test)[:,1]
    rows.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'ROC_AUC':roc_auc_score(y_test,prob)})
results=pd.DataFrame(rows).sort_values('ROC_AUC',ascending=False).reset_index(drop=True)
display(results)

## Business interpretation
Use the segment tables to identify associations that may be useful for retention analysis. Do not interpret association as proof of causation. The model metrics above are calculated from the current dataset at runtime.

## Power BI handoff
Recommended KPIs: churn rate, customers, churned customers, average monthly charges and model ROC-AUC. Recommended visuals: churn by contract, satisfaction, internet service, tenure and support calls.